# Ejercicio 5: Espacio Vectorial

## Objetivo de la práctica
- Implementar un Sistema de Recuperación de Información completo, desde la lectura del corpus hasta la recuperación de resultados.

Nombre: Joel Patricio Quilumba Morocho

## Parte 0: Carga del Corpus

Vamos a utilizar la API de Kaggle para acceder al dataset _Wikipedia Text Corpus for NLP and LLM Projects_

El corpus está disponible desde este [link](https://www.kaggle.com/datasets/gzdekzlkaya/wikipedia-text-corpus-for-nlp-and-llm-projects?utm_source=chatgpt.com)

### Actividad

1. Carga el corpus
2. Realiza las etapas de preprocesamiento sobre el corpus


In [3]:
import os
import re
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)


# Actividad 1: Carga del Corpus
ruta_archivo = r"C:\Users\LabP5E004\Downloads\05ev\wikipedia_text_corpus.csv"

print("Cargando el corpus desde el CSV...")
# Carga inicial de 10,000 filas para optimizar pruebas
df = pd.read_csv(ruta_archivo, nrows=10000)
print(f"Corpus cargado: {len(df)} documentos.")


# Actividad 2: Preprocesamiento
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocesar(texto):
    if not isinstance(texto, str):
        return ""
    
    # Normalización a minúsculas y extracción de tokens alfabéticos
    tokens = re.findall(r'[a-z]+', texto.lower())
    
    # Eliminación de palabras funcionales y lematización
    tokens_limpios = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words]
    
    # Unión de tokens para compatibilidad con TfidfVectorizer
    return " ".join(tokens_limpios)

print("Preprocesando textos...")
columna_texto = 'text' if 'text' in df.columns else df.columns[0]

# Aplicación de preprocesamiento sobre el DataFrame
df['texto_preprocesado'] = df[columna_texto].apply(preprocesar)

# Conversión a lista para vectorización posterior
corpus_textos = df['texto_preprocesado'].tolist()

print("Preprocesamiento completado.")
print(df[[columna_texto, 'texto_preprocesado']].head())

Cargando el corpus desde el CSV...
Corpus cargado: 10000 documentos.
Preprocesando textos...
Preprocesamiento completado.
                                                text  \
0  Anovo\n\nAnovo (formerly A Novo) is a computer...   
1  Battery indicator\n\nA battery indicator (also...   
2  Bob Pease\n\nRobert Allen Pease (August 22, 19...   
3  CAVNET\n\nCAVNET was a secure military forum w...   
4  CLidar\n\nThe CLidar is a scientific instrumen...   

                                  texto_preprocesado  
0  anovo anovo formerly novo computer service com...  
1  battery indicator battery indicator also known...  
2  bob pea robert allen pea august june analog in...  
3  cavnet cavnet secure military forum became ope...  
4  clidar clidar scientific instrument used measu...  


## Parte 1: Recuperación con TF-IDF

### Actividad:
3. Obtén la representación vectorial de los documentos utilizando el modelo TF-IDF
4. A partir de un conjunto de 10 queries, verifica la recuperación del sistema

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# ACTIVIDAD 3: REPRESENTACIÓN VECTORIAL (TF-IDF)

# Inicializar y ajustar el vectorizador con el corpus preprocesado
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(df['texto_preprocesado'])

vocabulario = vectorizer.get_feature_names_out()
print("Matriz TF-IDF")
print(f"Total de Documentos (N): {tfidf_matrix.shape[0]}")
print(f"Dimensiones (Vocabulario): {tfidf_matrix.shape[1]}")

# ACTIVIDAD 4: RECUPERACIÓN CON 10 QUERIES 

# Listado de 10 queries
queries = [
    "Computer services company in Beauvais France",
    "Electric vehicle battery state of charge",
    "History of artificial intelligence and machine learning",
    "Space exploration missions to Mars",
    "Climate change impacts on Arctic ice",
    "Olympic games gold medals history",
    "Ancient Roman empire architecture and engineering",
    "Quantum mechanics and particle physics",
    "Cybersecurity threats and data encryption",
    "World War II strategic battles in Europe"
]

def recuperar_resultados(query):
    # Preprocesar la consulta reutilizando la funcion previa
    query_limpia = preprocesar(query)
    
    # Transformar la consulta al espacio vectorial
    query_vec = vectorizer.transform([query_limpia])
    
    # Calcular la similitud de coseno
    similitudes = cosine_similarity(query_vec, tfidf_matrix).flatten()
    
    # Obtener el indice del mejor resultado
    idx_mejor = similitudes.argsort()[-1]
    score_mejor = similitudes[idx_mejor]
    
    return idx_mejor, score_mejor

print("\n--------------------------------------------------")
print("VERIFICACION DEL SISTEMA DE RECUPERACION")
print("--------------------------------------------------\n")

# Identificar la columna de texto original
columna_texto = 'text' if 'text' in df.columns else df.columns[0]

for i, q in enumerate(queries, 1):
    idx, score = recuperar_resultados(q)
    
    print(f"Query {i}: '{q}'")
    if score > 0:
        # Extraemos un fragmento del texto original para mostrar la relevancia
        texto_recuperado = str(df[columna_texto].iloc[idx])[:180].replace('\n', ' ')
        print(f"Top Result [ID: {idx}] (Similitud: {score:.4f})")
        print(f"Contenido: {texto_recuperado}...")
    else:
        print("No se encontraron documentos relevantes.")
    print("-" * 60)

Matriz TF-IDF
Total de Documentos (N): 10000
Dimensiones (Vocabulario): 113191

--------------------------------------------------
VERIFICACION DEL SISTEMA DE RECUPERACION
--------------------------------------------------

Query 1: 'Computer services company in Beauvais France'
Top Result [ID: 0] (Similitud: 0.3334)
Contenido: Anovo  Anovo (formerly A Novo) is a computer services company based in Beauvais, France. It was founded in 1987, went public in 1999, and is currently a member of the CAC Small.  I...
------------------------------------------------------------
Query 2: 'Electric vehicle battery state of charge'
Top Result [ID: 993] (Similitud: 0.6650)
Contenido: Battery electric vehicle  A battery electric vehicle (BEV), pure electric vehicle or all-electric vehicle is a type of electric vehicle (EV) that uses chemical energy stored in rec...
------------------------------------------------------------
Query 3: 'History of artificial intelligence and machine learning'
Top Resul

## Parte 2: Recuperación con BM25

### Actividad:
5. Implementa un sistema de recuperación usando el modelo BM25.
6. Para el mismo conjunto de 10 queries, verifica la recuperación del sistema

In [5]:
from rank_bm25 import BM25Okapi
import numpy as np

# PARTE 2: RECUPERACION CON BM25
print("Iniciando configuracion del modelo BM25...")

# Tokenizar el corpus (BM25 requiere listas de palabras)
tokenized_corpus = [str(doc).split() for doc in df['texto_preprocesado']]

# Inicializar el modelo BM25
bm25 = BM25Okapi(tokenized_corpus)

print("Modelo BM25 inicializado y calibrado con el corpus actual.\n")

# VERIFICACION DEL SISTEMA DE RECUPERACION
queries = [
    "Computer services company in Beauvais France",
    "Electric vehicle battery state of charge",
    "History of artificial intelligence and machine learning",
    "Space exploration missions to Mars",
    "Climate change impacts on Arctic ice",
    "Olympic games gold medals history",
    "Ancient Roman empire architecture and engineering",
    "Quantum mechanics and particle physics",
    "Cybersecurity threats and data encryption",
    "World War II strategic battles in Europe"
]

print("-" * 75)
print("RESULTADOS DE RECUPERACION - ALGORITMO BM25")
print("-" * 75)

# Identificar columna de texto para visualizacion
columna_texto = 'text' if 'text' in df.columns else df.columns[0]

# Evaluar consultas
for i, q in enumerate(queries, 1):
    
    # Preprocesar y tokenizar la consulta
    query_limpia = preprocesar(q)
    tokenized_query = query_limpia.split()
    
    # Calcular puntajes de similitud
    doc_scores = bm25.get_scores(tokenized_query)
    
    # Obtener el documento mas relevante (Top 1)
    idx_mejor = np.argmax(doc_scores)
    score_mejor = doc_scores[idx_mejor]
    
    # Mostrar resultados
    print(f"Consulta [{i}/10]: {q}")
    print(f"Vectores procesados: {tokenized_query}")
    
    if score_mejor > 0:
        # Extraer fragmento de 160 caracteres
        texto_recuperado = str(df[columna_texto].iloc[idx_mejor])[:160].replace('\n', ' ')
        print(f"  > Documento ID recuperado : {idx_mejor}")
        print(f"  > Puntaje BM25 (Score)    : {score_mejor:.4f}")
        print(f"  > Extracto del documento  : {texto_recuperado}...")
    else:
        print("  > Resultado: No se encontraron documentos relevantes para los vectores ingresados.")
        
    print("-" * 75)

Iniciando configuracion del modelo BM25...
Modelo BM25 inicializado y calibrado con el corpus actual.

---------------------------------------------------------------------------
RESULTADOS DE RECUPERACION - ALGORITMO BM25
---------------------------------------------------------------------------
Consulta [1/10]: Computer services company in Beauvais France
Vectores procesados: ['computer', 'service', 'company', 'beauvais', 'france']
  > Documento ID recuperado : 0
  > Puntaje BM25 (Score)    : 23.9688
  > Extracto del documento  : Anovo  Anovo (formerly A Novo) is a computer services company based in Beauvais, France. It was founded in 1987, went public in 1999, and is currently a member ...
---------------------------------------------------------------------------
Consulta [2/10]: Electric vehicle battery state of charge
Vectores procesados: ['electric', 'vehicle', 'battery', 'state', 'charge']
  > Documento ID recuperado : 993
  > Puntaje BM25 (Score)    : 23.7134
  > Extracto del

## Parte 3: Comparación de resultados

### Actividad:
7. Verifica cuáles documentos son recuperados (y en qué orden) por cada modelo de recuperación 

In [6]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

print("Iniciando modulo de comparacion analitica de modelos de recuperacion...")
print("-" * 80)

# 1. Definicion de parametros de evaluacion
K = 3 

# 2. Iteracion comparativa sobre el conjunto de consultas
for i, q in enumerate(queries, 1):
    
    print(f"\n[CONSULTA {i}]: {q}")
    
    #  Normalizacion unificada de la consulta
    query_limpia = preprocesar(q)
    tokenized_query = query_limpia.split()
    
    # ---------------------------------------------------
    # 3. RECUPERACION CON TF-IDF (Modelo Vectorial Clásico)
    # ---------------------------------------------------
    query_vector = vectorizer.transform([query_limpia])
    tfidf_scores = cosine_similarity(query_vector, tfidf_matrix).flatten()
    
    # Obtener los indices de los K documentos con mayor similitud coseno
    # Se utiliza argsort, se toman los ultimos K y se invierte el arreglo (orden descendente)
    tfidf_top_indices = np.argsort(tfidf_scores)[-K:][::-1]
    
    # ---------------------------------------------------
    # 4. RECUPERACION CON BM25 (Modelo Probabilistico)
    # ---------------------------------------------------
    bm25_scores = bm25.get_scores(tokenized_query)
    
    # Obtener los indices de los K documentos con mayor puntaje BM25
    bm25_top_indices = np.argsort(bm25_scores)[-K:][::-1]
    
    # ---------------------------------------------------
    # 5. DESPLIEGUE COMPARATIVO DE RESULTADOS
    # ---------------------------------------------------
    print(f"{'Ranking':<10} | {'TF-IDF (Doc_ID - Score)':<30} | {'BM25 (Doc_ID - Score)'}")
    print("-" * 80)
    
    for rank in range(K):
        # Datos del TF-IDF
        idx_tfidf = tfidf_top_indices[rank]
        score_tfidf = tfidf_scores[idx_tfidf]
        
        # Datos del BM25
        idx_bm25 = bm25_top_indices[rank]
        score_bm25 = bm25_scores[idx_bm25]
        
        # Formateo condicional para evitar imprimir resultados con score 0.0
        str_tfidf = f"Doc: {idx_tfidf:<5} (Score: {score_tfidf:.4f})" if score_tfidf > 0 else "Sin coincidencias"
        str_bm25 = f"Doc: {idx_bm25:<5} (Score: {score_bm25:.4f})" if score_bm25 > 0 else "Sin coincidencias"
        
        # Impresion de la fila comparativa
        print(f"Top {rank + 1:<6} | {str_tfidf:<30} | {str_bm25}")
        
print("\n" + "=" * 80)
print("FIN DEL ANALISIS COMPARATIVO")

Iniciando modulo de comparacion analitica de modelos de recuperacion...
--------------------------------------------------------------------------------

[CONSULTA 1]: Computer services company in Beauvais France
Ranking    | TF-IDF (Doc_ID - Score)        | BM25 (Doc_ID - Score)
--------------------------------------------------------------------------------
Top 1      | Doc: 0     (Score: 0.3334)     | Doc: 0     (Score: 23.9688)
Top 2      | Doc: 3176  (Score: 0.1667)     | Doc: 7909  (Score: 10.5902)
Top 3      | Doc: 4553  (Score: 0.1660)     | Doc: 2632  (Score: 10.3934)

[CONSULTA 2]: Electric vehicle battery state of charge
Ranking    | TF-IDF (Doc_ID - Score)        | BM25 (Doc_ID - Score)
--------------------------------------------------------------------------------
Top 1      | Doc: 993   (Score: 0.6650)     | Doc: 993   (Score: 23.7134)
Top 2      | Doc: 4159  (Score: 0.5683)     | Doc: 4159  (Score: 22.3925)
Top 3      | Doc: 1391  (Score: 0.5378)     | Doc: 1     (Score